# 🎬 HyperFlow H3 Flash Studio - Interactive Web UI (Human Mode)
### Fast Joint Video & 32 kHz Audio Reference Generation via MiniMax H3 Flash LoRA (Ref2VA INT8)
* **Model:** `VideoRebirthAI/hyperflow-h3-flash` (ref2va_int8_convrot_comfyui)
* **Accelerator:** Kaggle Dual Nvidia Tesla T4 GPU (or P100)
* **Mode:** **Interactive Browser UI** (Launches ComfyUI + Public Cloudflare Web Link)


In [ ]:
# [Step 1/4] Environment & ComfyUI Setup
import os, sys, subprocess, time

try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

t_total_start = time.time()
t_step1_start = time.time()

print('=' * 75, flush=True)
print('>>> [1/4] Installing ComfyUI & HyperFlow Custom Nodes...', flush=True)
print('=' * 75, flush=True)

# Install Cloudflared
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

# Clone ComfyUI core
if not os.path.exists('ComfyUI'):
    subprocess.run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git', shell=True, check=True)

# Clone HyperFlow node
custom_nodes_path = 'ComfyUI/custom_nodes/ComfyUI-HyperFlow-Flash'
if not os.path.exists(custom_nodes_path):
    subprocess.run('git clone --depth 1 https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git ' + custom_nodes_path, shell=True, check=True)

pkgs = ['huggingface_hub>=0.28.0', 'hf_transfer', 'gguf', 'pillow', 'requests', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ComfyUI/requirements.txt'], check=True)
t_step1 = time.time() - t_step1_start
print(f'⏱️ [TIMING] Step 1 (ComfyUI & Dependencies Setup): {t_step1:.2f}s', flush=True)


In [ ]:
# [Step 2/4] High-Speed Model Ingestion (Mounted Dataset + HF Fallback)
import os, shutil, glob, time
from huggingface_hub import hf_hub_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
t_step2_start = time.time()

print('=' * 75, flush=True)
print('>>> [2/4] Provisioning Model Checkpoints...', flush=True)
print('=' * 75, flush=True)

os.makedirs('ComfyUI/models/diffusion_models', exist_ok=True)
os.makedirs('ComfyUI/models/text_encoders', exist_ok=True)
os.makedirs('ComfyUI/models/vae', exist_ok=True)

ds_candidates = glob.glob('/kaggle/input/*minimax*weights*')
ds_root = ds_candidates[0] if ds_candidates else '/kaggle/input/minimax-h3-open-weights'
print(f'Detected Mounted Dataset Root: {ds_root} (Exists: {os.path.exists(ds_root)})', flush=True)

def link_or_fetch(rel_ds_path, local_dst, hf_repo, hf_file):
    t_start_item = time.time()
    src = os.path.join(ds_root, rel_ds_path)
    if os.path.exists(src):
        if not os.path.exists(local_dst):
            try:
                os.symlink(src, local_dst)
            except Exception:
                shutil.copyfile(src, local_dst)
        dur = time.time() - t_start_item
        print(f'✓ [MOUNTED 0-SEC] Linked in {dur:.3f}s: {local_dst}', flush=True)
    else:
        if not os.path.exists(local_dst):
            print(f'⏳ [DOWNLOADING FROM HF] {hf_file}...', flush=True)
            p = hf_hub_download(repo_id=hf_repo, filename=hf_file)
            try:
                os.symlink(p, local_dst)
            except Exception:
                shutil.copyfile(p, local_dst)
            dur = time.time() - t_start_item
            size_gb = os.path.getsize(local_dst) / (1024**3)
            print(f'✓ [DOWNLOADED] {local_dst} ({size_gb:.2f} GB) in {dur:.2f}s', flush=True)
        else:
            print(f'✓ [CACHED] Already present: {local_dst}', flush=True)

link_or_fetch('clip/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'ComfyUI/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
link_or_fetch('vae/minimax_h3_video_vae_int8_convrot.safetensors', 'ComfyUI/models/vae/minimax_h3_video_vae_fp16.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors')
link_or_fetch('vae/minimax_h3_audio_vae_fp32.safetensors', 'ComfyUI/models/vae/minimax_h3_audio_vae_fp32.safetensors', 'Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors')
link_or_fetch('diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors', 'ComfyUI/models/diffusion_models/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors', 'VideoRebirthAI/hyperflow-h3-flash', 'ref2va_int8_convrot_comfyui/minimax_h3_ref2va_hyperflow_flash_14b-int8_convrot.safetensors')
t_step2 = time.time() - t_step2_start
print(f'⏱️ [TIMING] Step 2 (Total Model Provisioning Time): {t_step2:.2f}s', flush=True)


In [ ]:
# [Step 3/4] Ingest Example References & Setup Workflows
import os, shutil, urllib.request

input_dir = "ComfyUI/input"
os.makedirs(input_dir, exist_ok=True)

official_urls = {
    "ref_image_1.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_ref2v_int8.jpg",
    "ref_image_2.png": "https://raw.githubusercontent.com/Video-Rebirth/hyperflow-h3-flash/main/integrations/ComfyUI-HyperFlow-Flash/example_workflows/hyperflow_i2v_int8.jpg"
}
for fname, url in official_urls.items():
    dst = os.path.join(input_dir, fname)
    if not os.path.exists(dst):
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as e:
            print(f"Notice on {fname}: {e}")

print("✓ Reference input assets ready in ComfyUI/input/", flush=True)


In [ ]:
# [Step 4/4] Start ComfyUI Server & Cloudflare Live Web UI
import subprocess, threading, time, re, urllib.request

PORT = 8188
t_step4_start = time.time()
print('=' * 75, flush=True)
print('>>> [4/4] Launching Interactive ComfyUI Server & Cloudflare Tunnel...', flush=True)
print('=' * 75, flush=True)

comfy_proc = subprocess.Popen(f'{sys.executable} ComfyUI/main.py --listen 127.0.0.1 --port {PORT} --enable-cors-header', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

def monitor():
    for line in comfy_proc.stdout:
        if any(w in line for w in ['Loaded', 'Prompt', 'Executing', 'HyperFlow', 'error', 'Error']):
            print(f'[ComfyUI] {line.strip()}', flush=True)

threading.Thread(target=monitor, daemon=True).start()

tunnel_proc = subprocess.Popen(f'/usr/local/bin/cloudflared tunnel --url http://127.0.0.1:{PORT}', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
t_tunnel = time.time()
while time.time() - t_tunnel < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

t_step4 = time.time() - t_step4_start
t_total_ready = time.time() - t_total_start

print('\n' + '=' * 80, flush=True)
if public_url:
    print(f'🌐 COMFYUI LIVE WEB UI: {public_url}', flush=True)
    print('👉 Open the link above in your browser to interact with the ComfyUI canvas!', flush=True)
print('=' * 80, flush=True)
print(f'⏱️ [TIMING] Server Launch & Tunnel Ready: {t_step4:.2f}s', flush=True)
print(f'⏱️ [TIMING] 🔥 TOTAL TIME FROM BOOT TO LIVE URL: {t_total_ready:.2f}s ({t_total_ready/60:.2f} min)', flush=True)
print('=' * 80 + '\n', flush=True)

counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 5 == 0:
        print(f'[Heartbeat] ComfyUI interactive server is online ({counter} min).', flush=True)
